---
# API-First Design in Python Using FastAPI
---

This tutorial shows you:
- How to use an API-first in Python, i.e. design your API contract first, then implement it in Python instead of hacking endpoints and *documenting later*.
- How to use common tools.
- A concrete workflow.
- This example uses [FastAPI](https://fastapi.tiangolo.com) as the backend and a Python client app.

What *API-first/design-first* means (in practice)
1. Start from an API contract (usually OpenAPI / Swagger).
2. Get agreement from stakeholders (backend, frontend, mobile, other services).
3. Generate mocks, docs, and even server/client stubs from the contract.
4. Implement the Python code behind that contract.

Definitions in a nutshell
- API-first: treat APIs as primary products, designed from the ground up, not bolted on. 
- Design-first: focus on writing the specification before the code, collaboratively. 
- OpenAPI is the usual lingua franca.

Typical API-first toolchain in Python (core building blocks)
1. API description language
   - [OpenAPI 3.x (YAML/JSON)](https://spec.openapis.org/oas) – standard way to describe REST APIs. 
2. Design and collaboration tools (give you visual editing, mocking, reviews), e.g.
   - [Stoplight Studio](https://stoplight.io) ([GitHub](https://github.com/stoplightio/studio), [Desktop](https://github.com/stoplightio/desktop/releases))
   - [Swagger Editor](https://editor.swagger.io)
   - [Postman](https://www.postman.com) ([VSCode Extension](https://marketplace.visualstudio.com/items?itemName=Postman.postman-for-vscode))
   - [OpenAPI (Swagger) Editor VSCode Extension](https://marketplace.visualstudio.com/items?itemName=42Crunch.vscode-openapi)
   - [Swagger Viewer VSCode Extension](https://marketplace.visualstudio.com/items?itemName=Arjun.swagger-viewer)
3. Code generators
   - [openapi-generator](https://openapi-generator.tech) tool ([GitHub](https://github.com/OpenAPITools/openapi-generator)), [swagger-codegen](https://swagger.io/tools/swagger-codegen) ([GitHub](https://github.com/swagger-api/swagger-codegen)), [datamodel-code-generator](https://docs.pydantic.dev/latest/integrations/datamodel_code_generator) ([GitHub](https://github.com/koxudaxi/datamodel-code-generator)), etc. generate:
     - Python server stubs
     - Python client SDKs
     - Pydantic models, tests, etc. 
4. Python frameworks
   - [FastAPI](https://fastapi.tiangolo.com) ([GitHub](https://github.com/fastapi)) – modern, async, type-hint heavy, OpenAPI native (auto docs). 
   - [Connexion](https://connexion.readthedocs.io/en/latest) ([GitHub](https://github.com/spec-first/connexion)) on top of [Flask](https://flask.palletsprojects.com/en/stable) – spec-first, drives routing directly from an OpenAPI file. 
GitHub
   - [Flask](https://flask.palletsprojects.com/en/stable) / [Django](https://www.djangoproject.com) REST Framework can also be used with design-first, but need more glue.

---
## Install VSCode Extensions

Tools:
- [OpenAPI (Swagger) Editor](https://marketplace.visualstudio.com/items?itemName=42Crunch.vscode-openapi)
  - Gives you:
    - YAML/JSON schema validation
    - IntelliSense/autocomplete
    - Error highlighting
  - Installs an OpenAPI view (icon) in VSCode's left margin.
- [Swagger Viewer](https://marketplace.visualstudio.com/items?itemName=Arjun.swagger-viewer)
  - Gives you:
    - Render button / command to see docs UI
    - Preview of OpenAPI 2.0/3.0 inside VS Code or browser
  - Open an OpenAPI YAML file in VSCode's editor, then click the OpenAPI preview button (top right) to preview it.

In [ ]:
!code --install-extension 42crunch.vscode-openapi --force
!code --install-extension arjun.swagger-viewer --force

Installing extensions...
Extension '42crunch.vscode-openapi' is already installed.
Installing extensions...
Extension 'arjun.swagger-viewer' is already installed.


---
## Install the [openapi-generator](https://openapi-generator.tech) tool ([GitHub](https://github.com/OpenAPITools/openapi-generator))

- On Ubuntu, you would do this, via VSCode's built-in terminal, like this:

    ```bash
    sudo apt update && sudo apt install -y build-essential procps curl file git

    NONINTERACTIVE=1 /bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

    echo 'eval "$(/home/linuxbrew/.linuxbrew/bin/brew shellenv)"' >> ~/.bashrc
    eval "$(/home/linuxbrew/.linuxbrew/bin/brew shellenv)"

    brew install openapi-generator

    brew --version
    openapi-generator version
    ```

- You can also install the tool via npm:

    ```bash
    npm install @openapitools/openapi-generator-cli -g
    ```

In [ ]:
!npm install @openapitools/openapi-generator-cli -g

---
## Concrete API-first workflow (step by step)

Let’s assume you’re building a small service in Python.

### Step 0 – Create a workspace folder and a Python virtual environment

Steps:
- Create a new folder `fastapi`
- In VSCode's built-in termial (in the `fastapi` folder), run the following commands (assuming you have [miniconda](https://www.anaconda.com/docs/getting-started/miniconda/main) installed)

```bash
conda create -y -p ./.conda python=3.12
conda activate ./.conda
python -m pip install --upgrade pip
```

### Step 1 – Design the API spec

Steps:
- Create a new file `openapi.yaml` in the `fastapi` folder.
- Replace its contents with the code below.
  - It specifies basic CRUD-functionality for managing orders.

```yaml
openapi: 3.0.3
info:
  title: Orders API
  version: 1.0.0
servers:
  - url: http://localhost:8080
paths:
  /orders:
    get:
      summary: List orders
      operationId: list_orders
      tags: [Orders]
      responses:
        '200':
          description: A list of orders
          content:
            application/json:
              schema:
                type: array
                items:
                  $ref: '#/components/schemas/Order'
    post:
      summary: Create an order
      operationId: create_order
      tags: [Orders]
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/NewOrder'
      responses:
        '201':
          description: Created order
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Order'
        '400':
          description: Bad request
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Error'

  /orders/{order_id}:
    get:
      summary: Get a single order
      operationId: get_order
      tags: [Orders]
      parameters:
        - name: order_id
          in: path
          required: true
          schema:
            type: string
      responses:
        '200':
          description: An order
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Order'
        '404':
          description: Not found
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Error'

    put:
      summary: Update an existing order (full replace)
      operationId: update_order
      tags: [Orders]
      parameters:
        - name: order_id
          in: path
          required: true
          schema:
            type: string
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: '#/components/schemas/NewOrder'
      responses:
        '200':
          description: Updated order
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Order'
        '400':
          description: Bad request
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Error'
        '404':
          description: Not found
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Error'

    delete:
      summary: Delete an order
      operationId: delete_order
      tags: [Orders]
      parameters:
        - name: order_id
          in: path
          required: true
          schema:
            type: string
      responses:
        '204':
          description: Order deleted
        '404':
          description: Not found
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/Error'

components:
  schemas:
    Order:
      type: object
      required: [id, item, quantity]
      properties:
        id:
          type: string
          example: "1"
        item:
          type: string
          example: "Coffee"
        quantity:
          type: integer
          example: 2

    NewOrder:
      type: object
      required: [item, quantity]
      properties:
        item:
          type: string
        quantity:
          type: integer
          minimum: 1

    Error:
      type: object
      required: [message]
      properties:
        message:
          type: string
```

You can:
- edit this in a visual editor (e.g. open the file `openapi.yaml` in VSCode's editor, then click on the icon for the installed VSCode extension `OpenAPI` in the left side bar), or
- as raw YAML in VSCode's editor.
- share it with frontend / mobile / other teams and iterate before coding.

### Step 2 – Generate server + models

Run the code below in VSCode's built-in terminal (in the folder `fastapi`).
- This example uses a Connexion/Flask style server.
- This gives you:
  - routing code wired to your paths
  - data models (possibly Pydantic / dataclasses depending on generator)
  - stub functions where you put real logic 
- You then only implement the internals.
- The code generates the server in the subfolder `orders_server` within the `fastapi` folder.

```bash
openapi-generator-cli generate \
  -i openapi.yaml \
  -g python-fastapi \
  -o orders_server
```

**Note:** If on Windows, replace `\` with `` ` `` (PowerShell) or `^` (CMD) in the code above.

### Step 3 – Implement handlers in Python
- Fastapi reads `openapi.yaml` at startup and maps /orders $\rightarrow$ these handler functions. 
- Your contract remains the source of truth.
- In a Fastapi-style project, stubs for your controllers are generated in `orders_server/src/apis`.
- Open the file `orders_server/src/apis/orders_api.py` and replace its contents as below.

```python
# coding: utf-8

from typing import Dict, List

from fastapi import (
    APIRouter,
    HTTPException,
)

from openapi_server.models.error import Error
from openapi_server.models.new_order import NewOrder
from openapi_server.models.order import Order

# In-memory "database"
ORDERS_DB: Dict[str, Order] = {}
NEXT_ID: int = 1

router = APIRouter()


@router.get(
    "/orders",
    response_model=List[Order],
    responses={
        200: {"model": List[Order], "description": "A list of orders"},
    },
    tags=["default"],
    summary="List orders",
    response_model_by_alias=True,
)
async def list_orders() -> List[Order]:
    """GET /orders"""
    return list(ORDERS_DB.values())


@router.post(
    "/orders",
    status_code=201,  # <-- important: matches OpenAPI + client expectation
    response_model=Order,
    responses={
        201: {"model": Order, "description": "Created order"},
        400: {"model": Error, "description": "Bad request"},
    },
    tags=["default"],
    summary="Create an order",
    response_model_by_alias=True,
)
async def create_order(new_order: NewOrder) -> Order:
    """POST /orders"""
    global NEXT_ID

    if new_order.quantity is None or new_order.quantity < 1:
        raise HTTPException(status_code=400, detail="`quantity` must be >= 1")

    order_id = str(NEXT_ID)
    NEXT_ID += 1

    order = Order(
        id=order_id,
        item=new_order.item,
        quantity=new_order.quantity,
    )
    ORDERS_DB[order_id] = order
    return order


@router.get(
    "/orders/{order_id}",
    status_code=200,
    response_model=Order,
    responses={
        200: {"model": Order, "description": "An order"},
        404: {"model": Error, "description": "Not found"},
    },
    tags=["default"],
    summary="Get a single order",
    response_model_by_alias=True,
)
async def get_order(order_id: str) -> Order:
    """GET /orders/{order_id}"""
    order = ORDERS_DB.get(order_id)
    if not order:
        raise HTTPException(status_code=404, detail=f"Order {order_id} not found")
    return order


@router.put(
    "/orders/{order_id}",
    status_code=200,
    response_model=Order,
    responses={
        200: {"model": Order, "description": "Updated order"},
        400: {"model": Error, "description": "Bad request"},
        404: {"model": Error, "description": "Not found"},
    },
    tags=["default"],
    summary="Update an existing order (full replace)",
    response_model_by_alias=True,
)
async def update_order(order_id: str, new_order: NewOrder) -> Order:
    """PUT /orders/{order_id}"""
    if order_id not in ORDERS_DB:
        raise HTTPException(status_code=404, detail=f"Order {order_id} not found")

    if new_order.quantity is None or new_order.quantity < 1:
        raise HTTPException(status_code=400, detail="`quantity` must be >= 1")

    updated = Order(
        id=order_id,
        item=new_order.item,
        quantity=new_order.quantity,
    )
    ORDERS_DB[order_id] = updated
    return updated


@router.delete(
    "/orders/{order_id}",
    status_code=204,  # <-- important: matches OpenAPI (No Content)
    responses={
        204: {"description": "Order deleted"},
        404: {"model": Error, "description": "Not found"},
    },
    tags=["default"],
    summary="Delete an order",
    response_model_by_alias=True,
)
async def delete_order(order_id: str) -> None:
    """DELETE /orders/{order_id}"""
    if order_id not in ORDERS_DB:
        raise HTTPException(status_code=404, detail=f"Order {order_id} not found")

    del ORDERS_DB[order_id]
    # 204 No Content: return None, FastAPI sends empty body
    return None
```

### Step 5 – Install dependencies and start the server

Run the commands below in VSCode's built-in terminal (in the folder `fastapi/orders_server`).
- Assumes you have [curl](https://curl.se) ([GitHub](https://github.com/curl/curl)) installed.
- You can also use e.g. [Postman](https://www.postman.com) ([VSCode Extension](https://marketplace.visualstudio.com/items?itemName=Postman.postman-for-vscode)).

**Note:** If on Windows, replace `\` with `` ` `` (PowerShell) or `^` (CMD) in the code above.

```bash
# Make sure you are in the correct Python environment
conda activate ../.conda

# Dependencies (python packages) were added by openai-generator in requirements.txt
pip install -r requirements.txt

# Change to the "src" folder
cd src

# Start the server (the entry point is in the file "src/openapi-server/main.py)")
# - see the README.md file in the fastapi/orders_server folder
uvicorn openapi_server.main:app --host 0.0.0.0 --port 8080
```

### Step 6 – Verify the API is working as expected

Run the code below in a new VSCode built-in terminal.
- Assumes you have [curl](https://curl.se) ([GitHub](https://github.com/curl/curl)) installed.
- You can also use e.g. [Postman](https://www.postman.com) ([VSCode Extension](https://marketplace.visualstudio.com/items?itemName=Postman.postman-for-vscode)).

**Note:** If on Windows, replace `\` with `` ` `` (PowerShell) or `^` (CMD) in the code above.

```bash
# Get all orders (empty)
curl http://localhost:8080/orders

# Create (an order with item:Coffee, quantity:2)
curl -X POST http://localhost:8080/orders \
  -H "Content-Type: application/json" \
  -d '{"item": "Coffee", "quantity": 2}'

# Get all orders (one order with id 1)
curl http://localhost:8080/orders

# Update orders/1 (PUT) - change item:Espresso quantity:1
curl -X PUT http://localhost:8080/orders/1 \
  -H "Content-Type: application/json" \
  -d '{"item": "Espresso", "quantity": 1}'

# Get orders/1
curl http://localhost:8080/orders/1

# Delete orders/1 
curl -X DELETE http://localhost:8080/orders/1 -v

# Verify delete (get orders/1) -> 404 (not found)
curl http://localhost:8080/orders/1

# Get all orders (empty)
curl http://localhost:8080/orders
```

### Step 7 – Generate client SDKs

Run the code below in VSCode's built-in terminal (in the folder `fastapi`).
- Now you have a strongly-typed Python client you can reuse in other services or scripts.
- The code generates the client library in the subfolder `orders_client_library` within the `fastapi` folder.

```bash
openapi-generator-cli generate \
  -i openapi.yaml \
  -g python \
  -o orders_client_library
```

**Note:** If on Windows, replace `\` with `` ` `` (PowerShell) or `^` (CMD) in the code above.

### Step 8 – Install the client library into the Python environment

Run the commands below in VSCode's built-in terminal (in the folder `fastapi/orders_client_library`).
- The command `pip install -e .` installs the `orders_client_library` as a Python package.
- The `-e` flag stands for `editable`, which means the Python packge is editable (used during development).

```bash
# Make sure you are in the correct Python environment
conda activate ../.conda

# Install the orders_client_library
pip install -e .
```

### Step 9 – Create a client application that uses the client library

Steps:
- Create a subfolder `client` in the folder `fastapi`.
- In the `client` folder, create a file `example_client.py` and replace its contents as below.

```python
"""
Example of using the generated client SDK to talk to our Orders API.

Prerequisites:
- The server is running on http://localhost:8080
- The client library has been generated and installed:

    openapi-generator generate \
        -i openapi.yaml \
        -g python \
        -o orders_client_library

    cd orders_client_library
    pip install -e .

Run this file from the `client/` directory:
    cd client
    python example_client.py
"""

from __future__ import annotations

from openapi_client import Configuration, ApiClient
from openapi_client.api.orders_api import OrdersApi
from openapi_client.models.new_order import NewOrder


def main() -> None:
    # Configure the client to talk to our local server
    config = Configuration(
        host="http://localhost:8080",
    )

    # Use ApiClient as a context manager so it cleans up properly
    with ApiClient(config) as api_client:
        api = OrdersApi(api_client)

        # 1. List existing orders
        print("=== LIST (initial) ===")
        initial_orders = api.list_orders()
        print(initial_orders)

        # 2. Create a new order
        print("\n=== CREATE ===")
        new_order = NewOrder(item="Coffee", quantity=2)
        created_order = api.create_order(new_order)
        print("Created:", created_order)

        order_id = created_order.id

        # 3. Get the created order
        print("\n=== GET ===")
        fetched = api.get_order(order_id)
        print("Fetched:", fetched)

        # 4. Update the order (full replace via PUT)
        print("\n=== UPDATE ===")
        updated_body = NewOrder(item="Espresso", quantity=1)
        updated = api.update_order(order_id, updated_body)
        print("Updated:", updated)

        # 5. Confirm the update via GET
        print("\n=== GET (after update) ===")
        fetched_after_update = api.get_order(order_id)
        print("Fetched after update:", fetched_after_update)

        # 6. Delete the order
        print("\n=== DELETE ===")
        api.delete_order(order_id)
        print(f"Deleted order {order_id}")

        # 7. Final list to confirm deletion
        print("\n=== LIST (final) ===")
        final_orders = api.list_orders()
        print(final_orders)


if __name__ == "__main__":
    main()
```

### Step 10 – Run the client application

Run the command below in VSCode's built-in terminal (in the folder `fastapi/client`).
1. Creates an order.
2. Fetches the order.
3. Updates the order.
4. Fecthes the order again.
5. Deletes the order.
6. Lists all orders.

```bash
python example_client.py
```

### Step 11 – Stop the server

In the terminal running the server, press:
- Windows/Linux: `Ctrl + C`
- MacOS: `Cmd + C`

### Step 12 – Create a server outside the generated `fastapi/orders_server`

Steps:
- Create a Python library of the generated server `orders_server`:

- Run the commands below in VSCode's built-in terminal (in the folder `fastapi/orders_server`).
  - The command `pip install -e .` installs the `orders_server` as a Python package.
  - The `-e` flag stands for `editable`, which means the Python packge is editable (used during development).

    ```bash
    # Make sure you are in the correct Python environment
    conda activate ../.conda

    # Install the orders_server as a library
    pip install -e .
    ```
- Create a subfolder `server` in the folder `fastapi`.
- In the `server` folder, create a file `main.py` and replace its contents as below.

    ```python
    from fastapi import FastAPI

    # This comes from the generated server package
    from openapi_server.apis.orders_api import router as orders_router

    app = FastAPI(
        title="Orders API (custom host)",
        version="1.0.0",
    )

    # Reuse the generated router (with your CRUD logic) in your own app
    app.include_router(orders_router)
    ```

### Step 13 – Run the server and the client

- Run the command below in VSCode's built-in terminal (in the folder `fastapi/server`).

    ```bash
    uvicorn main:app --host 0.0.0.0 --port 8080
    ```

- Run the command below in a new VSCode built-in terminal (in the folder `fastapi/client`).

    ```bash
    python example_client.py
    ```

- In the terminal running the server, press:
  - Windows/Linux: `Ctrl + C`
  - MacOS: `Cmd + C`